# 04 Relation Probe

역할: `position_shift`에서 absolute position score는 흔들리고, scale-normalized relation score는 안정적인지 먼저 작은 sanity check로 확인한다.


## Cell Role: Scope

처음 범위는 `breakfast_box / position_shift / high`로 제한한다. 이 노트북은 SAM 기반 query component matching 전에, known transform만 사용해서 geometry score의 기대 동작을 검증한다.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'

def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )

colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    git_cmd = (
        f'if [ -d "{colab_checkout}/.git" ]; then git -C "{colab_checkout}" pull --ff-only; '
        f'else git clone "{REPO_URL}" "{colab_checkout}"; fi'
    )
    print(git_cmd)
    subprocess.run(['bash', '-lc', git_cmd], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and is_regram_root(p)), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        'ReGraM checkout not found. In Colab, clone/pull the repo to /content/ReGraM first, '
        'or run 01_run_orchestrator setup cells before this notebook.'
    )
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for source_root in (SRC_ROOT, SRC_ROOT / 'relation'):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Probe Config

Relation probe의 최소 실행 단위를 정한다. `LIMIT`은 처음에는 작게 두고, 결과 형태가 맞으면 전체 normal로 늘린다.


In [ ]:
CATEGORY = 'breakfast_box'
SEVERITY = 'high'
LIMIT = 30
MAX_COMPONENTS = 8

POSITION_MANIFEST = REPO_ROOT / 'manifests' / 'query_position_shift.jsonl'
OUTPUT_PATH = EXP_ROOT / 'reports' / 'relation_probe' / f'{CATEGORY}_position_shift_{SEVERITY}_known_transform.json'
RAW_CATEGORY_GOOD = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection' / CATEGORY / 'test' / 'good'

required_paths = {
    'relation runner': SRC_ROOT / 'relation' / 'run_relation_probe.py',
    'position manifest': POSITION_MANIFEST,
    'raw LOCO test/good images': RAW_CATEGORY_GOOD,
}
missing_paths = {name: path for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(f'Missing required relation-probe path(s): {missing_paths}')

display(pd.DataFrame([{
    'category': CATEGORY,
    'severity': SEVERITY,
    'limit': LIMIT,
    'max_components': MAX_COMPONENTS,
    'manifest': str(POSITION_MANIFEST),
    'output': str(OUTPUT_PATH),
}]))


## Cell Role: Run Known-Transform Probe

Clean image에서 lightweight proxy component를 추출하고, `position_shift`의 known transform을 centroid에 적용한다. 여기서 보는 것은 detection 성능이 아니라 score family의 geometry sanity check다.


In [ ]:
cmd = [
    sys.executable,
    str(SRC_ROOT / 'relation' / 'run_relation_probe.py'),
    '--repo-root', str(REPO_ROOT),
    '--manifest', str(POSITION_MANIFEST),
    '--category', CATEGORY,
    '--severity', SEVERITY,
    '--limit', str(LIMIT),
    '--max-components', str(MAX_COMPONENTS),
    '--output', str(OUTPUT_PATH),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Cell Role: Inspect Scores

기대 패턴은 `s_abs`, `s_centered_raw`, `s_pair_raw`는 커지고, `s_centered_norm`, `s_pair_norm`은 0에 가깝게 유지되는 것이다.


In [ ]:
summary = json.loads(OUTPUT_PATH.read_text())
metric_df = (
    pd.DataFrame([summary['metric_medians']])
    .T
    .reset_index()
    .rename(columns={'index': 'metric', 0: 'median'})
)
display(metric_df)

rows_df = pd.DataFrame(summary['rows'])
display(rows_df.head(10))

if summary['skipped_count']:
    display(pd.DataFrame(summary['skipped']).head(10))
